# Stage C / NB 14 — Conventional fusion baselines (E7a–E7e)

Protocol reference: experiment family **E7**; research question **RQ6**.

## Why this notebook decides the paper's central claim

NB 15 will run the LLM reasoner over the same agent outputs. The comparison that matters is
**E7f (reasoning) versus E7d (learned stacking)** — the same inputs, one aggregated by a language
model, the other by ridge regression. If reasoning does not beat stacking, the framework's claim
changes from *"reasoning improves accuracy"* to *"reasoning provides equivalent accuracy with
auditable, clinician-readable justification"*.

That second claim is still publishable, but only if this notebook gives stacking an honest
chance. **Do not tune the baselines downward.** A weak stacking baseline would manufacture a gap
and is precisely the criticism referee 2a levelled at the original submission.

## The arms

| arm | method |
| --- | --- |
| E7a | best single agent, selected on inner validation |
| E7b | unweighted mean and median over agent mRALE predictions |
| E7c | reliability-weighted mean — inverse-MAE, inverse-variance, and softmax(−MAE/τ) with a τ sweep |
| E7d | learned stacking: ridge and gradient-boosted trees on agent outputs plus uncertainty |
| E7e | learned gating: input-conditional agent weighting |

E7c's τ sweep is the direct answer to referee 1.1's question about the confidence-weighting
strategy — the rejected version asserted a weighting scheme without justifying it.

## Leakage discipline

Every weight, coefficient, gating model and τ is fitted on the **inner-validation** rows from
NB 13, then applied unchanged to the held-out fold. Nothing is fitted on test data. The gate
asserts it.

## Outputs (under `stage_C/nb14_fusion/`)
`fusion_predictions.jsonl`, `fusion_weights/`, `e7_fusion_metrics.csv`, `tau_sweep.csv`,
`stacking_coefficients.csv`, `usability.json`, `gate_nb14.json`.

## 1. Imports and the Stage A path contract

In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import re
import sys
import time
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Shared metric definitions live with the Stage B notebooks; every arm in Table 2 and every
# fusion/reasoner arm here must be scored by identical code or the comparison is invalid.
_SEARCH = [Path.cwd(), Path.cwd().parent, Path.cwd().parent / "stage_B",
           Path.cwd().parent.parent / "notebooks" / "stage_B"]
for _candidate in _SEARCH:
    if (_candidate / "cxr_metrics.py").is_file():
        sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError(f"cxr_metrics.py not found. Searched: {_SEARCH}")
import cxr_metrics as cm

SEED = 42
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

FALLBACK_STAGE_A = Path("/data/liangz2/openi/midrc/tetci_resubmit/stage_A")
for _candidate in [FALLBACK_STAGE_A / "nb00_environment" / "stage_a_paths.json",
                   Path.cwd() / "stage_a_paths.json",
                   Path.cwd().parent / "stage_A" / "nb00_environment" / "stage_a_paths.json"]:
    if _candidate.is_file():
        stage_paths = json.loads(_candidate.read_text(encoding="utf-8"))
        print("Path contract:", _candidate)
        break
else:
    raise FileNotFoundError("stage_a_paths.json not found. Run Stage A NB 00 first.")

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_ROOT = Path(stage_paths["stage_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
STAGE_B_DIR = STAGE_ROOT / "stage_B"
STAGE_C_DIR = STAGE_ROOT / "stage_C"
STAGE_C_DIR.mkdir(parents=True, exist_ok=True)
NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
NB04_DIR = Path(stage_paths["nb_output_dirs"]["nb04_localization"])
FOLD_DEF_DIR = NB02_DIR / "fold_definitions"
MODEL_REVISIONS = stage_paths.get("model_revisions", {})

N_FOLDS = 5
INTERNAL_VALIDATION_FRACTION = 0.10   # identical to Stage B, so inner splits match exactly

print("Stage C output:", STAGE_C_DIR)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2. Configuration

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler

NB14_DIR = STAGE_C_DIR / "nb14_fusion"
WEIGHTS_DIR = NB14_DIR / "fusion_weights"
for d in (NB14_DIR, WEIGHTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
NB13_DIR = STAGE_C_DIR / "nb13_registry"

TAU_GRID = [0.25, 0.5, 1.0, 2.0, 4.0]     # softmax(-MAE/tau) temperature, E7c
RIDGE_ALPHA = 1.0
GBT_PARAMS = {"n_estimators": 200, "max_depth": 3, "learning_rate": 0.05,
              "random_state": SEED}
N_BOOTSTRAP = 2000

# Fusion operates on the images EVERY participating agent scored. Reported explicitly, because
# a fusion arm fitted on the intersection is not comparable to a single agent scored on the
# whole cohort.
RESTRICT_TO_INTERSECTION = True

print("Output:", NB14_DIR)

## 3. Load the registry

In [ ]:
registry_path = NB13_DIR / "agent_registry.parquet"
if registry_path.is_file():
    try:
        registry = pd.read_parquet(registry_path)
    except Exception:
        registry = pd.read_csv(NB13_DIR / "agent_registry.csv")
elif (NB13_DIR / "agent_registry.csv").is_file():
    registry = pd.read_csv(NB13_DIR / "agent_registry.csv")
else:
    raise FileNotFoundError(
        f"No registry at {NB13_DIR}. Run NB 13 first — it is what guarantees the reliability "
        "statistics used here were computed on inner validation only.")

reliability = pd.read_csv(NB13_DIR / "agent_reliability_inner.csv")
manifest = json.loads((NB13_DIR / "registry_manifest.json").read_text(encoding="utf-8"))
print(f"Registry: {len(registry):,} rows  agents={sorted(registry['agent'].unique())}")
print(f"Fold-specific arms: {json.dumps(manifest.get('selected_arms_per_fold', {}), indent=2)[:400]}")

PROTOCOL_AGENTS = [a for a in ["A1", "A2", "A3", "A4", "A5", "A6"]
                   if a in set(registry["agent"])]
INNER_FOLD_COLUMNS = [f"inner_fold_{k}" for k in range(N_FOLDS)]
SELECTED_FOLD_COLUMNS = [f"selected_for_fold_{k}" for k in range(N_FOLDS)]
missing_columns = [c for c in INNER_FOLD_COLUMNS + SELECTED_FOLD_COLUMNS
                   + ["selected_for_outer_fold"]
                   if c not in registry.columns]
if missing_columns:
    raise RuntimeError(
        f"The registry lacks {missing_columns}. Re-run NB 13: fold-relative inner-validation "
        "membership cannot be recovered from a single `split` column, and without it every "
        "fitting set here would be empty.")

outer_selected = registry[registry.get("selected_for_outer_fold", False) == True].copy()
def agents_with_endpoint(column, minimum=0.90):
    agents = []
    for agent in PROTOCOL_AGENTS:
        rows = outer_selected[outer_selected["agent"] == agent]
        coverage = float(rows[column].notna().mean()) if len(rows) and column in rows else 0.0
        if coverage >= minimum:
            agents.append(agent)
        else:
            print(f"  {agent} excluded from {column} fusion: coverage {coverage:.1%}")
    return agents

MRALE_AGENTS = agents_with_endpoint("mrale_total")
COVID_AGENTS = agents_with_endpoint("covid_score")
AGENTS = MRALE_AGENTS  # existing E7a-E7e mRALE implementation below
if len(MRALE_AGENTS) < 2:
    raise RuntimeError(f"mRALE fusion needs at least two numeric agents; found {MRALE_AGENTS}.")
if len(COVID_AGENTS) < 2:
    raise RuntimeError(f"COVID fusion needs at least two continuous-score agents; found {COVID_AGENTS}.")
print(f"\nmRALE agents: {MRALE_AGENTS}")
print(f"COVID agents: {COVID_AGENTS}")
print(f"Evidence-only for mRALE: {sorted(set(PROTOCOL_AGENTS) - set(MRALE_AGENTS))}")

META_COLUMNS = ["fold", "group_id", "split", "gt_mrale_total",
                "gt_covid", "severity_band"] + INNER_FOLD_COLUMNS
meta = registry.drop_duplicates("image_key").set_index("image_key")[META_COLUMNS]

def endpoint_frame(fold, agents, value_column):
    chosen = registry[(registry[f"selected_for_fold_{fold}"] == True)
                      & registry["agent"].isin(agents)].copy()
    values = chosen.pivot_table(index="image_key", columns="agent",
                                values=value_column, aggfunc="first")
    missing = sorted(set(agents) - set(values.columns))
    if missing:
        raise RuntimeError(f"fold {fold}: selected {value_column} columns missing for {missing}")
    out = meta.join(values, how="inner")
    if value_column == "mrale_total":
        uncertainty = chosen.pivot_table(index="image_key", columns="agent",
                                         values="mrale_uncertainty", aggfunc="first")
        out = out.join(uncertainty.add_prefix("unc_"), how="left")
        for agent in agents:
            if f"unc_{agent}" not in out.columns:
                out[f"unc_{agent}"] = np.nan
    return out.dropna(subset=agents) if RESTRICT_TO_INTERSECTION else out

MRALE_FRAMES = {k: endpoint_frame(k, MRALE_AGENTS, "mrale_total")
                for k in range(N_FOLDS)}
COVID_FRAMES = {k: endpoint_frame(k, COVID_AGENTS, "covid_score")
                for k in range(N_FOLDS)}
frame = pd.concat([MRALE_FRAMES[k][MRALE_FRAMES[k]["fold"] == k]
                   for k in range(N_FOLDS)]).sort_index()
if not frame.index.is_unique:
    raise RuntimeError("Fold-specific mRALE application frame contains duplicate images.")
print(f"Fusion denominator: {len(frame):,} fold-specific complete cases")
print("Fitting/application rows by fold:")
for _fold in range(N_FOLDS):
    _frame = MRALE_FRAMES[_fold]
    _fit = int((_frame[f"inner_fold_{_fold}"] == 1).sum())
    _apply = int((_frame["fold"] == _fold).sum())
    print(f"  fold {_fold}: fit {_fit:,}, apply {_apply:,}")
    if not _fit or not _apply:
        raise RuntimeError(f"fold {_fold}: empty fitting/application set after endpoint filtering")

## 4. Fitting helpers — inner validation only

Every arm receives the same two row sets per fold: `fit` (inner-validation rows from the other
folds) and `apply` (that fold's held-out rows). No arm sees the second while fitting.

In [ ]:
def fold_partition(fold):
    """
    Rows used to FIT fold `fold`'s weights, and the rows those weights are applied to.

    Fitting rows are the ones NB 13 flagged as fold `fold`'s inner validation. By construction
    they come from the other four folds, so no weight is ever fitted on the data it scores.
    The `fold != fold` clause below is redundant given that construction and is kept as a
    belt-and-braces filter -- it costs nothing and would catch a corrupted registry.
    """
    fold_frame = MRALE_FRAMES[fold]
    fit = fold_frame[(fold_frame[f"inner_fold_{fold}"] == 1)
                     & (fold_frame["fold"] != fold)]
    apply_to = fold_frame[fold_frame["fold"] == fold]
    return fit, apply_to


def agent_inner_mae(fit_rows):
    """Per-agent MAE on the fitting rows only. This is what every weight derives from."""
    out = {}
    truth = fit_rows["gt_mrale_total"].astype(float).to_numpy()
    for agent in AGENTS:
        pred = fit_rows[agent].astype(float).to_numpy()
        mask = ~np.isnan(pred)
        out[agent] = float(np.abs(pred[mask] - truth[mask]).mean()) if mask.any() else float("inf")
    return out


def weights_from_mae(mae, scheme, tau=1.0):
    agents = list(mae)
    values = np.array([mae[a] for a in agents], dtype=float)
    if scheme == "inverse_mae":
        w = 1.0 / np.clip(values, 1e-6, None)
    elif scheme == "inverse_variance":
        w = 1.0 / np.clip(values ** 2, 1e-6, None)
    elif scheme == "softmax":
        z = -values / max(tau, 1e-6)
        z = z - z.max()
        w = np.exp(z)
    elif scheme == "uniform":
        w = np.ones_like(values)
    else:
        raise ValueError(scheme)
    w = w / w.sum()
    return dict(zip(agents, w.tolist()))


def apply_weights(rows, weights):
    matrix = rows[list(weights)].astype(float).to_numpy()
    w = np.array([weights[a] for a in weights], dtype=float)
    mask = ~np.isnan(matrix)
    weighted = np.where(mask, matrix, 0.0) @ w
    norm = mask.astype(float) @ w
    return np.where(norm > 0, weighted / np.clip(norm, 1e-9, None), np.nan)


MRALE_MIN, MRALE_MAX = 0, 24


def to_mrale(value):
    """
    Round a continuous fusion output to an admissible mRALE total, or None if unusable.

    Ridge and gradient-boosted regressors are unconstrained and routinely emit values just
    below zero on the easiest cases. An mRALE of -0.3 is not a prediction, it is an artefact
    of an unbounded model, and letting it through crashes the quadratic-weighted-kappa
    histogram (which cannot index a negative rating). Clipping to the scale's own range is the
    only defensible reading of such an output; `n_clipped` below reports how often it happened
    so the clipping is visible rather than hidden.
    """
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None, False
    rounded = int(round(float(value)))
    clipped = max(MRALE_MIN, min(MRALE_MAX, rounded))
    return clipped, clipped != rounded


def score_predictions(keys, predicted, label):
    rows, n_clipped = [], 0
    for key, value in zip(keys, predicted):
        truth = frame.loc[key]
        total, was_clipped = to_mrale(value)
        n_clipped += int(was_clipped)
        rows.append({"gt_mrale_total": int(truth["gt_mrale_total"]), "mrale_total": total})
    metrics = cm.mrale_metrics(rows)
    metrics["n_clipped_to_scale"] = n_clipped
    if n_clipped:
        print(f"    {label}: {n_clipped} prediction(s) fell outside 0-24 and were clipped")
    return metrics, rows

## 5. E7a–E7c — single agent, simple ensembles, reliability weighting

In [ ]:
predictions_by_arm = defaultdict(dict)     # arm -> {image_key: prediction}
weights_log, tau_rows = [], []

for fold in range(N_FOLDS):
    fit, apply_to = fold_partition(fold)
    if not len(fit) or not len(apply_to):
        print(f"  fold {fold}: skipped (fit={len(fit)}, apply={len(apply_to)})")
        continue
    mae = agent_inner_mae(fit)
    keys = apply_to.index.tolist()

    # --- E7a: best single agent, chosen on the fitting rows -------------------------------
    best_agent = min(mae, key=mae.get)
    predictions_by_arm["E7a_best_single"].update(
        dict(zip(keys, apply_to[best_agent].astype(float).to_numpy())))
    weights_log.append({"fold": fold, "arm": "E7a_best_single",
                        "detail": json.dumps({"chosen": best_agent,
                                              "inner_mae": {k: round(v, 4) for k, v in mae.items()}})})

    # --- E7b: unweighted mean and median ---------------------------------------------------
    matrix = apply_to[AGENTS].astype(float).to_numpy()
    predictions_by_arm["E7b_mean"].update(dict(zip(keys, np.nanmean(matrix, axis=1))))
    predictions_by_arm["E7b_median"].update(dict(zip(keys, np.nanmedian(matrix, axis=1))))

    # --- E7c: reliability weighting, with the tau sweep -----------------------------------
    for scheme in ["inverse_mae", "inverse_variance"]:
        w = weights_from_mae(mae, scheme)
        predictions_by_arm[f"E7c_{scheme}"].update(dict(zip(keys, apply_weights(apply_to, w))))
        weights_log.append({"fold": fold, "arm": f"E7c_{scheme}",
                            "detail": json.dumps({k: round(v, 4) for k, v in w.items()})})
    for tau in TAU_GRID:
        w = weights_from_mae(mae, "softmax", tau)
        arm = f"E7c_softmax_tau{tau}"
        predictions_by_arm[arm].update(dict(zip(keys, apply_weights(apply_to, w))))
        weights_log.append({"fold": fold, "arm": arm,
                            "detail": json.dumps({k: round(v, 4) for k, v in w.items()})})
        tau_rows.append({"fold": fold, "tau": tau,
                         **{f"w_{k}": round(v, 4) for k, v in w.items()}})

pd.DataFrame(weights_log).to_csv(WEIGHTS_DIR / "fold_weights.csv", index=False)
pd.DataFrame(tau_rows).to_csv(NB14_DIR / "tau_sweep.csv", index=False)
print(f"E7a-E7c complete: {len([a for a in predictions_by_arm if a.startswith('E7')])} arms")

## 6. E7d–E7e — learned stacking and gating

The features are the agents' mRALE predictions, their per-agent uncertainty where available, and
simple dispersion statistics. Gating adds the same features but learns an input-conditional
weight per agent rather than a single blend.

These are the baselines the reasoner has to beat. They are given uncertainty and dispersion
features precisely so that the comparison in NB 15 is against a competent opponent.

In [ ]:
def build_features(rows):
    matrix = rows[AGENTS].astype(float).to_numpy()
    filled = np.where(np.isnan(matrix), np.nanmean(matrix, axis=1, keepdims=True), matrix)
    features = [filled]
    uncertainty_columns = [f"unc_{a}" for a in AGENTS if f"unc_{a}" in rows.columns]
    unc = rows[uncertainty_columns].astype(float).to_numpy() \
        if uncertainty_columns else np.zeros((len(rows), 0))
    if unc.size:
        features.append(np.nan_to_num(unc, nan=0.0))
    # Dispersion: how much the agents disagree is itself informative.
    features.append(np.nanstd(matrix, axis=1, keepdims=True))
    features.append((np.nanmax(matrix, axis=1) - np.nanmin(matrix, axis=1)).reshape(-1, 1))
    features.append(np.isnan(matrix).sum(axis=1, keepdims=True).astype(float))
    return np.nan_to_num(np.hstack(features), nan=0.0)


FEATURE_NAMES = (AGENTS
                 + [f"unc_{a}" for a in AGENTS]
                 + ["disagreement_std", "disagreement_range", "n_missing"])

stacking_rows = []
for fold in range(N_FOLDS):
    fit, apply_to = fold_partition(fold)
    if len(fit) < 30 or not len(apply_to):
        continue
    X_fit, y_fit = build_features(fit), fit["gt_mrale_total"].astype(float).to_numpy()
    X_apply = build_features(apply_to)
    keys = apply_to.index.tolist()
    scaler = StandardScaler().fit(X_fit)

    ridge = Ridge(alpha=RIDGE_ALPHA).fit(scaler.transform(X_fit), y_fit)
    predictions_by_arm["E7d_ridge_stack"].update(
        dict(zip(keys, ridge.predict(scaler.transform(X_apply)))))
    for name, coef in zip(FEATURE_NAMES, ridge.coef_):
        stacking_rows.append({"fold": fold, "model": "ridge", "feature": name,
                              "coefficient": round(float(coef), 5)})

    gbt = GradientBoostingRegressor(**GBT_PARAMS).fit(X_fit, y_fit)
    predictions_by_arm["E7d_gbt_stack"].update(dict(zip(keys, gbt.predict(X_apply))))
    for name, imp in zip(FEATURE_NAMES, gbt.feature_importances_):
        stacking_rows.append({"fold": fold, "model": "gbt", "feature": name,
                              "coefficient": round(float(imp), 5)})

    # --- E7e gating: predict per-agent weights from the input ------------------------------
    # Target: which agent was closest on each fitting row. A classifier over that target gives
    # an input-conditional weighting, which is what "gating" means here.
    truth_fit = fit["gt_mrale_total"].astype(float).to_numpy()
    errors = np.abs(np.nan_to_num(fit[AGENTS].astype(float).to_numpy(),
                                  nan=1e9) - truth_fit[:, None])
    best_index = errors.argmin(axis=1)
    if len(set(best_index.tolist())) > 1:
        gate = LogisticRegression(max_iter=2000, random_state=SEED)
        gate.fit(scaler.transform(X_fit), best_index)
        proba = gate.predict_proba(scaler.transform(X_apply))
        classes = [AGENTS[i] for i in gate.classes_]
        matrix = apply_to[classes].astype(float).to_numpy()
        mask = ~np.isnan(matrix)
        weighted = np.where(mask, matrix, 0.0) * proba
        norm = mask.astype(float) * proba
        gated = weighted.sum(axis=1) / np.clip(norm.sum(axis=1), 1e-9, None)
        predictions_by_arm["E7e_gating"].update(dict(zip(keys, gated)))
        weights_log.append({"fold": fold, "arm": "E7e_gating",
                            "detail": json.dumps({"mean_weight":
                                {a: round(float(p), 4) for a, p in zip(classes, proba.mean(0))}})})

# --- COVID endpoint: the original implementation built covid_scores and never used it. ---
covid_predictions_by_arm = defaultdict(dict)
# Owner-fold inner scores are a separate artifact used only to choose NB 18 thresholds.
# They are never mixed with the outer-fold predictions above. The learned stack is fitted
# on this inner partition, so these are model-development scores (not performance estimates);
# their only allowed use is fixing an operating point before touching the outer fold.
covid_inner_predictions_by_arm = defaultdict(lambda: defaultdict(dict))
for fold in range(N_FOLDS):
    fold_frame = COVID_FRAMES[fold]
    fit = fold_frame[(fold_frame[f"inner_fold_{fold}"] == 1)
                     & (fold_frame["fold"] != fold)]
    apply_to = fold_frame[fold_frame["fold"] == fold]
    if len(fit) < 30 or not len(apply_to):
        continue
    y_fit = (fit["gt_covid"] == "Yes").astype(int).to_numpy()
    if len(set(y_fit.tolist())) < 2:
        continue
    keys = apply_to.index.tolist()
    inner_keys = fit.index.tolist()
    brier = {a: float(np.mean((fit[a].astype(float).to_numpy() - y_fit) ** 2))
             for a in COVID_AGENTS}
    best_agent = min(brier, key=brier.get)
    covid_predictions_by_arm["E7a_best_single"].update(
        dict(zip(keys, apply_to[best_agent].astype(float).to_numpy())))
    covid_inner_predictions_by_arm["E7a_best_single"][fold].update(
        dict(zip(inner_keys, fit[best_agent].astype(float).to_numpy())))
    matrix = apply_to[COVID_AGENTS].astype(float).to_numpy()
    inner_matrix = fit[COVID_AGENTS].astype(float).to_numpy()
    covid_predictions_by_arm["E7b_mean"].update(dict(zip(keys, matrix.mean(axis=1))))
    covid_predictions_by_arm["E7b_median"].update(dict(zip(keys, np.median(matrix, axis=1))))
    covid_inner_predictions_by_arm["E7b_mean"][fold].update(
        dict(zip(inner_keys, inner_matrix.mean(axis=1))))
    covid_inner_predictions_by_arm["E7b_median"][fold].update(
        dict(zip(inner_keys, np.median(inner_matrix, axis=1))))
    for scheme in ["inverse_mae", "inverse_variance"]:
        weights = weights_from_mae(brier, scheme)
        covid_predictions_by_arm[f"E7c_{scheme}"].update(
            dict(zip(keys, apply_weights(apply_to, weights))))
        covid_inner_predictions_by_arm[f"E7c_{scheme}"][fold].update(
            dict(zip(inner_keys, apply_weights(fit, weights))))
    for tau in TAU_GRID:
        weights = weights_from_mae(brier, "softmax", tau)
        covid_predictions_by_arm[f"E7c_softmax_tau{tau}"].update(
            dict(zip(keys, apply_weights(apply_to, weights))))
        covid_inner_predictions_by_arm[f"E7c_softmax_tau{tau}"][fold].update(
            dict(zip(inner_keys, apply_weights(fit, weights))))

    X_fit = fit[COVID_AGENTS].astype(float).to_numpy()
    X_apply = apply_to[COVID_AGENTS].astype(float).to_numpy()
    scaler = StandardScaler().fit(X_fit)
    logistic = LogisticRegression(max_iter=2000, random_state=SEED).fit(
        scaler.transform(X_fit), y_fit)
    covid_predictions_by_arm["E7d_logistic_stack"].update(dict(zip(
        keys, logistic.predict_proba(scaler.transform(X_apply))[:, 1])))
    covid_inner_predictions_by_arm["E7d_logistic_stack"][fold].update(dict(zip(
        inner_keys, logistic.predict_proba(scaler.transform(X_fit))[:, 1])))
    gbt_covid = GradientBoostingClassifier(**GBT_PARAMS).fit(X_fit, y_fit)
    covid_predictions_by_arm["E7d_gbt_stack"].update(
        dict(zip(keys, gbt_covid.predict_proba(X_apply)[:, 1])))
    covid_inner_predictions_by_arm["E7d_gbt_stack"][fold].update(
        dict(zip(inner_keys, gbt_covid.predict_proba(X_fit)[:, 1])))

    errors = np.abs(X_fit - y_fit[:, None])
    best_index = errors.argmin(axis=1)
    if len(set(best_index.tolist())) > 1:
        gate = LogisticRegression(max_iter=2000, random_state=SEED).fit(
            scaler.transform(X_fit), best_index)
        gate_probability = gate.predict_proba(scaler.transform(X_apply))
        classes = gate.classes_.astype(int)
        gated = (X_apply[:, classes] * gate_probability).sum(axis=1)
        covid_predictions_by_arm["E7e_gating"].update(dict(zip(keys, gated)))
        inner_gate_probability = gate.predict_proba(scaler.transform(X_fit))
        inner_gated = (X_fit[:, classes] * inner_gate_probability).sum(axis=1)
        covid_inner_predictions_by_arm["E7e_gating"][fold].update(
            dict(zip(inner_keys, inner_gated)))

# Persist the owner-fold score contract consumed by NB 18. One file may contain several
# arms; NB 18 filters by `arm` and verifies fold ownership plus NB 13 membership.
for fold in range(N_FOLDS):
    inner_rows = []
    for arm, by_fold in sorted(covid_inner_predictions_by_arm.items()):
        for key, raw_score in sorted(by_fold.get(fold, {}).items()):
            truth_row = COVID_FRAMES[fold].loc[key]
            score = float(np.clip(raw_score, 0.0, 1.0))
            inner_rows.append(cm.make_prediction_row(
                image_key=str(key), cohort="MIDRC", subcohort="MIDRC",
                filename=str(key).split("::")[-1], held_out_fold=fold,
                agent="FUSION", arm=arm, view="registry",
                task="covid_classification",
                covid_pred="Yes" if score >= 0.5 else "No",
                covid_score=score, gt_covid=str(truth_row["gt_covid"]),
                gt_mrale_total=int(truth_row["gt_mrale_total"]), valid=True,
                model_id="fusion", model_revision=None,
                extra={"score_role": "threshold_selection_only",
                       "owner_fold": fold,
                       "actual_data_fold": int(truth_row["fold"]),
                       "membership": f"inner_fold_{fold}",
                       "fit_scope": "same_inner_partition"}))
    fold_dir = NB14_DIR / "folds" / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    cm.write_jsonl(fold_dir / "inner_validation_score_records.jsonl", inner_rows)
    print(f"  fold {fold}: wrote {len(inner_rows):,} owner-produced inner COVID scores")

pd.DataFrame(stacking_rows).to_csv(NB14_DIR / "stacking_coefficients.csv", index=False)
pd.DataFrame(weights_log).to_csv(WEIGHTS_DIR / "fold_weights.csv", index=False)
print(f"E7d-E7e mRALE arms: {sorted(predictions_by_arm)}")
print(f"E7a-E7e COVID arms: {sorted(covid_predictions_by_arm)}")

## 7. Score every arm and identify the target NB 15 must beat

In [ ]:
fusion_rows, metric_rows = [], []
for arm, mapping in sorted(predictions_by_arm.items()):
    keys = sorted(mapping)
    values = [mapping[k] for k in keys]
    metrics, scored = score_predictions(keys, values, arm)
    metric_rows.append(OrderedDict([
        ("endpoint", "mrale"), ("arm", arm), ("n", len(keys)),
        ("mae", round(metrics.get("mae", float("nan")), 4)),
        ("rmse", round(metrics.get("rmse", float("nan")), 4)),
        ("qwk", round(metrics.get("qwk", float("nan")), 4)),
        ("spearman", round(metrics.get("spearman_rho", float("nan")), 4)),
        ("within1", round(metrics.get("within1_accuracy", float("nan")), 4)),
        ("n_clipped_to_scale", int(metrics.get("n_clipped_to_scale", 0))),
        ("mae_band_none", round(metrics.get("mae_band_none", float("nan")), 3)),
        ("mae_band_mild", round(metrics.get("mae_band_mild", float("nan")), 3)),
        ("mae_band_moderate", round(metrics.get("mae_band_moderate", float("nan")), 3)),
        ("mae_band_severe", round(metrics.get("mae_band_severe", float("nan")), 3)),
    ]))
    for key, value in zip(keys, values):
        truth = frame.loc[key]
        fusion_rows.append(cm.make_prediction_row(
            image_key=key, cohort="MIDRC", subcohort="MIDRC",
            filename=key.split("::")[-1], held_out_fold=int(truth["fold"]),
            agent="FUSION", arm=arm, view="registry", task="mrale_prediction",
            mrale_total=to_mrale(value)[0],
            mrale_total_expected=(None if value is None or (isinstance(value, float)
                                                            and math.isnan(value))
                                  else float(value)),
            gt_mrale_total=int(truth["gt_mrale_total"]), gt_covid=truth.get("gt_covid"),
            valid=not (value is None or (isinstance(value, float) and math.isnan(value))),
            model_id="fusion", model_revision=None))

# Score COVID fusion arms on their own endpoint-specific complete-case denominator.
for arm, mapping in sorted(covid_predictions_by_arm.items()):
    keys = sorted(mapping)
    scores = np.clip(np.asarray([mapping[k] for k in keys], dtype=float), 0.0, 1.0)
    truth = [str(meta.at[k, "gt_covid"]) for k in keys]
    decisions = ["Yes" if score >= 0.5 else "No" for score in scores]
    metrics = cm.classification_metrics(truth, decisions, scores.tolist())
    metric_rows.append(OrderedDict([
        ("endpoint", "covid"), ("arm", arm), ("n", len(keys)),
        ("auroc", round(metrics.get("auroc", float("nan")), 4)),
        ("auprc", round(metrics.get("auprc", float("nan")), 4)),
        ("balanced_accuracy", round(metrics.get("balanced_accuracy", float("nan")), 4)),
        ("sensitivity", round(metrics.get("sensitivity", float("nan")), 4)),
        ("specificity", round(metrics.get("specificity", float("nan")), 4)),
        ("mcc", round(metrics.get("mcc", float("nan")), 4)),
        ("brier", round(metrics.get("brier", float("nan")), 4)),
    ]))
    for key, score, decision in zip(keys, scores.tolist(), decisions):
        truth_row = meta.loc[key]
        fusion_rows.append(cm.make_prediction_row(
            image_key=key, cohort="MIDRC", subcohort="MIDRC",
            filename=key.split("::")[-1], held_out_fold=int(truth_row["fold"]),
            agent="FUSION", arm=arm, view="registry", task="covid_classification",
            covid_pred=decision, covid_score=float(score), gt_covid=truth_row["gt_covid"],
            gt_mrale_total=int(truth_row["gt_mrale_total"]), valid=True,
            model_id="fusion", model_revision=None))

cm.write_jsonl(NB14_DIR / "fusion_predictions.jsonl", fusion_rows)
e7 = pd.DataFrame(metric_rows)
e7.to_csv(NB14_DIR / "e7_fusion_metrics.csv", index=False)
pd.set_option("display.width", 200)
e7_mrale = e7[e7["endpoint"] == "mrale"].sort_values("mae")
e7_covid = e7[e7["endpoint"] == "covid"].sort_values("auroc", ascending=False)
print("mRALE fusion:")
print(e7_mrale[["arm", "n", "mae", "qwk", "spearman", "within1"]].to_string(index=False))
print("\nCOVID fusion:")
print(e7_covid[["arm", "n", "auroc", "auprc", "balanced_accuracy",
                "sensitivity", "specificity"]].to_string(index=False))

best_row = e7_mrale.iloc[0]
stack_rows = e7_mrale[e7_mrale["arm"].str.startswith("E7d")]
best_stack = stack_rows.iloc[0] if len(stack_rows) else best_row
best_covid = e7_covid.iloc[0]
covid_stack_rows = e7_covid[e7_covid["arm"].str.startswith("E7d")]
best_covid_stack = covid_stack_rows.iloc[0] if len(covid_stack_rows) else best_covid
cm.write_json(NB14_DIR / "e7f_target.json", {
    "best_fusion_arm": best_row["arm"], "best_fusion_mae": float(best_row["mae"]),
    "best_stacking_arm": best_stack["arm"], "best_stacking_mae": float(best_stack["mae"]),
    "best_covid_fusion_arm": best_covid["arm"],
    "best_covid_fusion_auroc": float(best_covid["auroc"]),
    "best_covid_stacking_arm": best_covid_stack["arm"],
    "best_covid_stacking_auroc": float(best_covid_stack["auroc"]),
    "n_images": int(best_row["n"]), "n_covid_images": int(best_covid["n"]),
    "denominator": "endpoint-specific all-agent intersection",
    "instruction": (
        "NB 15's E7f (LLM reasoning over the same agent outputs) must beat best_stacking_mae "
        "and best_covid_stacking_auroc on the SAME endpoint-specific images to support an "
        "accuracy claim for the corresponding endpoint. If it does not, the framework's "
        "contribution is auditability, not accuracy, and the paper should say so plainly "
        "rather than presenting a within-noise difference as a win."),
})
print()
print("=" * 78)
print(f"TARGET FOR NB 15: E7d stacking reaches MAE {best_stack['mae']:.4f} "
      f"({best_stack['arm']}) on {int(best_row['n']):,} images.")
print(f"COVID stacking target: AUROC {best_covid_stack['auroc']:.4f} "
      f"({best_covid_stack['arm']}) on {int(best_covid['n']):,} images.")
print("  The reasoner must beat these on the SAME endpoint-specific images, with paired CIs,")
print("  for an accuracy claim to hold. Do not weaken this baseline to create a gap.")

## 8. Gate

In [ ]:
failures, warnings = [], []

# The leakage assertion: nothing may be fitted on rows from the fold it is applied to.
for fold in range(N_FOLDS):
    fit, apply_to = fold_partition(fold)
    overlap = set(fit.index) & set(apply_to.index)
    if overlap:
        failures.append(f"fold {fold}: {len(overlap)} images used for BOTH fitting and "
                        "application. Every weight would be fitted on its own test data.")
    if len(fit) and (fit["fold"] == fold).any():
        failures.append(f"fold {fold}: fitting rows include the held-out fold itself.")
    covid_frame = COVID_FRAMES[fold]
    covid_fit = covid_frame[(covid_frame[f"inner_fold_{fold}"] == 1)
                            & (covid_frame["fold"] != fold)]
    covid_apply = covid_frame[covid_frame["fold"] == fold]
    if set(covid_fit.index) & set(covid_apply.index):
        failures.append(f"fold {fold}: COVID fitting/application rows overlap.")
if not failures:
    print("LEAKAGE ASSERTION PASSED: fitting and application sets are disjoint in all folds.")

# NB 18 must select each outer fold's COVID threshold from scores produced by that same
# fold's locked stacking model/development pipeline. Certify that contract here so a
# missing file is caught at the producer rather than much later in Stage D.
locked_covid_stack = str(best_covid_stack["arm"])
inner_contract_issues = []
for fold in range(N_FOLDS):
    path = NB14_DIR / "folds" / f"fold_{fold}" / "inner_validation_score_records.jsonl"
    rows = [row for row in (cm.read_jsonl(path) if path.is_file() else [])
            if str(row.get("arm")) == locked_covid_stack
            and int(row.get("held_out_fold", -1)) == fold]
    labels = {str(row.get("gt_covid")) for row in rows}
    if len(rows) < 5 or not {"Yes", "No"}.issubset(labels):
        inner_contract_issues.append((fold, len(rows), sorted(labels)))
if inner_contract_issues:
    failures.append(
        f"Owner-fold inner COVID scores are incomplete for {locked_covid_stack}: "
        f"{inner_contract_issues}. Re-run section 6 before Stage D.")
else:
    print(f"NB 18 INNER-SCORE CONTRACT PASSED: {locked_covid_stack}, all folds.")
small_inner_contract = []
for fold in range(N_FOLDS):
    path = NB14_DIR / "folds" / f"fold_{fold}" / "inner_validation_score_records.jsonl"
    n_locked = sum(str(row.get("arm")) == locked_covid_stack
                   for row in (cm.read_jsonl(path) if path.is_file() else []))
    if 5 <= n_locked < 20:
        small_inner_contract.append((fold, n_locked))
if small_inner_contract:
    warnings.append(
        f"Locked COVID threshold-development sets are small: {small_inner_contract}. "
        "NB 18 will report fold-to-fold threshold instability and confusion counts.")

covered = {row["image_key"] for row in fusion_rows
           if row.get("task") == "mrale_prediction"}
if RESTRICT_TO_INTERSECTION and len(covered) != len(frame):
    warnings.append(f"Fusion covers {len(covered):,} of {len(frame):,} intersection images.")

tau_frame = pd.read_csv(NB14_DIR / "tau_sweep.csv") if (NB14_DIR / "tau_sweep.csv").is_file() \
    else pd.DataFrame()
softmax_arms = e7_mrale[e7_mrale["arm"].str.contains("softmax")]
if len(softmax_arms) > 1:
    spread = float(softmax_arms["mae"].max() - softmax_arms["mae"].min())
    warnings.append(
        f"E7c tau sweep spans {spread:.4f} MAE across tau in {TAU_GRID}. "
        + ("Weighting temperature barely matters; report that rather than presenting a tuned "
           "tau as a design contribution." if spread < 0.1 else
           "Weighting temperature matters; report the selected tau and its inner-validation "
           "justification."))

simple = e7_mrale[e7_mrale["arm"].str.startswith(("E7a", "E7b"))]["mae"].min()
learned = e7_mrale[e7_mrale["arm"].str.startswith(("E7d", "E7e"))]["mae"].min()
if pd.notna(simple) and pd.notna(learned):
    warnings.append(
        f"Simple ensembling best MAE {simple:.4f} vs learned fusion best {learned:.4f} "
        f"(delta {learned - simple:+.4f}). "
        + ("Learned fusion does not beat a mean/median/best-single baseline, which bounds how "
           "much any aggregator can add on these agents — including the reasoner."
           if learned >= simple - 0.05 else
           "Learned fusion adds measurable value over simple ensembling."))

usability = {}
for arm in sorted(set(e7["arm"])):
    mrale_row = e7_mrale[e7_mrale["arm"] == arm]
    covid_row = e7_covid[e7_covid["arm"] == arm]
    usability[arm] = {
        "mrale_usable": bool(len(mrale_row)), "score_usable": bool(len(covid_row)),
        "n_mrale_images": int(mrale_row.iloc[0]["n"]) if len(mrale_row) else 0,
        "n_covid_images": int(covid_row.iloc[0]["n"]) if len(covid_row) else 0}
cm.write_json(NB14_DIR / "usability.json", usability)
cm.write_json(NB14_DIR / "run_config.json", {
    "written_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "14_conventional_fusion_baselines.ipynb",
    "protocol_experiments": ["E7a", "E7b", "E7c", "E7d", "E7e"],
    "protocol_agents": PROTOCOL_AGENTS, "mrale_agents": MRALE_AGENTS,
    "covid_agents": COVID_AGENTS, "seed": SEED,
    "tau_grid": TAU_GRID, "ridge_alpha": RIDGE_ALPHA, "gbt": GBT_PARAMS,
    "denominator": "endpoint-specific fold-selected agent intersection",
    "n_mrale_images": int(len(frame)),
    "n_covid_images": int(sum(len(COVID_FRAMES[k][COVID_FRAMES[k]["fold"] == k])
                              for k in range(N_FOLDS))),
    "fitting_scope": "inner_validation rows of the other folds, per fold",
    "inner_threshold_score_arm": locked_covid_stack,
    "inner_threshold_score_files": [
        str(NB14_DIR / "folds" / f"fold_{fold}" /
            "inner_validation_score_records.jsonl") for fold in range(N_FOLDS)],
    "inner_threshold_score_role": "model development / operating-point selection only",
    "note": ("These baselines must not be weakened to create a gap for E7f. A weak stacking "
             "baseline is exactly the criticism referee 2a levelled at the original."),
})


def report(title, messages):
    print(title)
    for m in messages or []:
        print("  -", m)
    if not messages:
        print("  none")


print()
report("WARNINGS", warnings)
print()
report("FAILURES", failures)
cm.write_json(NB14_DIR / "gate_nb14.json",
              {"passed": not failures, "failures": failures, "warnings": warnings})
if failures:
    detail = "\n".join(f"  [{i + 1}] {m}" for i, m in enumerate(failures))
    raise AssertionError(f"NB 14 gate failed with {len(failures)} blocking issue(s):\n{detail}")
print()
print("NB 14 gate: PASSED")

## Notes carried forward

- **`e7f_target.json` is what NB 15 must beat.** The comparison is E7f against the best *stacking*
  arm on the *same images*, with a paired confidence interval — not against the weakest fusion
  arm, and not on a different denominator.
- **The τ sweep answers referee 1.1 directly.** If MAE barely moves across τ, say so: the honest
  finding is that the weighting temperature does not matter, which is more useful than presenting
  a tuned value as a design contribution.
- **If learned fusion does not beat mean/median**, that bounds how much *any* aggregator can add
  on these agents — including the reasoner — and NB 15's result should be read in that light.
- Every weight, coefficient and gate is fitted on inner-validation rows only; the gate asserts the
  fitting and application sets are disjoint per fold.